# L17 — Machine Breakdowns and Preemptive Interrupts

**Module**: M06 | **Chapter**: 8 | **Lecture**: L17

## Learning Objectives
By the end of this notebook you will be able to:
1. Model random machine breakdowns using SimPy's `Interrupt` mechanism.
2. Implement preemptive resource interruption with job resumption.
3. Compute machine availability and its effect on throughput.
4. Explain the difference between preemptive and non-preemptive breakdown handling.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**
---

In [ ]:
import simpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as t_dist

## 1. Conceptual Model: Machine with Breakdowns

A single machine processes jobs. It fails randomly and must be repaired before processing resumes.

| Component | Distribution |
|---|---|
| Job inter-arrival | Exp(λ) |
| Job processing time | Exp(μ) |
| Time to failure (TTF) | Exp(1/MTTF) — time until next breakdown |
| Repair time | Exp(1/MTTR) |

**Machine availability**: A = MTTF / (MTTF + MTTR)

**Effect on throughput**: a machine that is down A fraction of the time has effective service rate μ·A.

In [ ]:
def breakdown_sim(lam: float, mu: float,
                  mttf: float, mttr: float,
                  sim_time: float = 100_000, seed: int = 0) -> dict:
    """
    M/M/1 queue with preemptive machine breakdowns.

    When a breakdown occurs mid-service, the job's remaining service
    time is paused and resumed after repair.
    """
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    machine = simpy.Resource(env, capacity=1)

    # Shared state for the breakdown process
    state = {
        'down': False,
        'n_failures': 0,
        'total_down': 0.0,
        'current_job_process': None,
    }
    waits, sojourns = [], []

    def job_process():
        arrival = env.now
        with machine.request() as req:
            yield req
            wait = env.now - arrival
            remaining = rng.exponential(1.0 / mu)
            state['current_job_process'] = env.active_process

            # Process in chunks — interrupted by breakdowns
            while remaining > 0:
                try:
                    start_chunk = env.now
                    yield env.timeout(remaining)
                    remaining = 0  # completed
                except simpy.Interrupt:
                    elapsed  = env.now - start_chunk
                    remaining -= elapsed
                    # Not interruptible during repair wait
                    state['current_job_process'] = None
                    while state['down']:
                        yield env.timeout(0.001)  # poll until repaired
                    # Re-register for future breakdowns if work remains
                    if remaining > 0:
                        state['current_job_process'] = env.active_process

            state['current_job_process'] = None

        waits.append(wait)
        sojourns.append(env.now - arrival)

    def breakdown_process():
        while True:
            # Time to next failure
            yield env.timeout(rng.exponential(mttf))
            state['n_failures'] += 1
            state['down'] = True
            t_fail = env.now

            # Interrupt job in progress (preemptive)
            if state['current_job_process'] is not None:
                state['current_job_process'].interrupt('breakdown')

            # Repair
            yield env.timeout(rng.exponential(mttr))
            state['total_down'] += env.now - t_fail
            state['down'] = False

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(job_process())

    env.process(arrivals())
    env.process(breakdown_process())
    env.run(until=sim_time)

    availability = 1.0 - state['total_down'] / sim_time
    return {
        'availability': availability,
        'n_failures': state['n_failures'],
        'Wq': np.mean(waits),
        'W':  np.mean(sojourns),
        'n_served': len(sojourns),
        'throughput': len(sojourns) / sim_time,
    }


# Parameters
lam, mu   = 3.0, 4.0
mttf_base = 20.0  # mean time to failure (hours)
mttr_base = 2.0   # mean time to repair (hours)
A_theory  = mttf_base / (mttf_base + mttr_base)

r = breakdown_sim(lam, mu, mttf_base, mttr_base, seed=42)
print(f"Breakdown simulation (λ={lam}, μ={mu}, MTTF={mttf_base}, MTTR={mttr_base})")
print(f"  Theoretical availability A = {A_theory:.4f}")
print(f"  Simulated availability    A = {r['availability']:.4f}")
print(f"  Mean wait in queue: {r['Wq']:.4f} hr = {r['Wq']*60:.1f} min")
print(f"  Mean sojourn:       {r['W']:.4f} hr")
print(f"  Failures: {r['n_failures']},  Served: {r['n_served']:,}")

## 2. Effect of Availability on Waiting Time

In [ ]:
# Fix MTTF+MTTR = 22 hr (same total cycle), vary fraction down
cycle_time = 22.0
avail_targets = [0.99, 0.95, 0.90, 0.80, 0.70]

rows = []
for A in avail_targets:
    mttf = A * cycle_time
    mttr = (1 - A) * cycle_time
    r = breakdown_sim(lam, mu, mttf, mttr, sim_time=200_000, seed=0)
    rows.append({'A_theory': A, 'A_sim': r['availability'],
                 'Wq_min': r['Wq']*60, 'W_min': r['W']*60,
                 'throughput': r['throughput']})

# Baseline (no breakdowns)
r_base = breakdown_sim(lam, mu, mttf=1e9, mttr=0.001, sim_time=200_000, seed=0)
print(f"No-breakdown baseline: Wq = {r_base['Wq']*60:.2f} min")
print()

df = pd.DataFrame(rows)
print(df.to_string(index=False, float_format='{:.3f}'.format))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(df['A_theory'], df['Wq_min'], 'o-', color='steelblue', lw=2)
ax.axhline(r_base['Wq']*60, color='gray', ls='--', lw=1, label='No-breakdown baseline')
ax.set_xlabel('Machine availability A = MTTF/(MTTF+MTTR)')
ax.set_ylabel('Mean wait in queue (min)')
ax.set_title(f'Effect of availability on Wq (λ={lam}, μ={mu})')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Non-Preemptive vs. Preemptive Breakdowns

**Non-preemptive**: the machine completes the current job before going down for repair.  
**Preemptive**: the machine stops immediately; the job resumes after repair.

In [ ]:
def nonpreemptive_breakdown_sim(lam, mu, mttf, mttr, sim_time=100_000, seed=0):
    """Non-preemptive breakdown: machine finishes current job before failing."""
    rng = np.random.default_rng(seed)
    env = simpy.Environment()
    machine = simpy.Resource(env, capacity=1)
    is_down = [False]
    total_down = [0.0]
    waits, sojourns = [], []

    def job_process():
        arrival = env.now
        with machine.request() as req:
            yield req
            wait = env.now - arrival
            # Wait if machine is down (non-preemptive: server unavailable)
            while is_down[0]:
                yield env.timeout(0.01)
            yield env.timeout(rng.exponential(1.0 / mu))
        waits.append(wait)
        sojourns.append(env.now - arrival)

    def breakdown_process():
        while True:
            yield env.timeout(rng.exponential(mttf))
            # Wait for current job to finish
            while machine.count > 0:
                yield env.timeout(0.01)
            is_down[0] = True
            t0 = env.now
            yield env.timeout(rng.exponential(mttr))
            total_down[0] += env.now - t0
            is_down[0] = False

    def arrivals():
        while True:
            yield env.timeout(rng.exponential(1.0 / lam))
            env.process(job_process())

    env.process(arrivals())
    env.process(breakdown_process())
    env.run(until=sim_time)
    return {'Wq': np.mean(waits), 'W': np.mean(sojourns),
            'availability': 1 - total_down[0]/sim_time}


lam_c, mu_c, mttf_c, mttr_c = 3.0, 4.0, 10.0, 2.0
pree   = breakdown_sim(lam_c, mu_c, mttf_c, mttr_c, seed=42)
nonpre = nonpreemptive_breakdown_sim(lam_c, mu_c, mttf_c, mttr_c, seed=42)

print(f"MTTF={mttf_c}, MTTR={mttr_c}, A_theory={mttf_c/(mttf_c+mttr_c):.3f}")
print(f"{'':20s}  {'Wq (min)':>10s}  {'W (min)':>10s}  {'Availability':>14s}")
print('-' * 60)
print(f"{'Preemptive':20s}  {pree['Wq']*60:>10.3f}  {pree['W']*60:>10.3f}  {pree['availability']:>14.4f}")
print(f"{'Non-preemptive':20s}  {nonpre['Wq']*60:>10.3f}  {nonpre['W']*60:>10.3f}  {nonpre['availability']:>14.4f}")

---
## Try It Yourself

1. **Multiple machines**: Extend to 3 machines sharing one repair technician. When a machine fails, it queues for the technician. Model the repair queue as a separate SimPy `Resource`. Compare availability under: (a) one technician, (b) two technicians.

2. **Scheduled preventive maintenance**: Replace random failures with a fixed PM interval (every 50 hr). At each PM, the machine goes down for a fixed 1-hr service. Compare total downtime and Wq for PM vs. corrective-only over T=10,000 hr.

3. **Reliability theory check**: With exponential TTF and TTR, the steady-state availability is A = MTTF/(MTTF+MTTR). Verify this formula against your simulation output for five different MTTF/MTTR combinations. Plot simulated A vs. theoretical A.